# Stance Training on Google Colab
Train the small two-stage stance pipeline on a Colab T4, then download the final checkpoint.

## What this notebook does
- clones your repo
- installs only the training dependencies
- builds `stage1_public_small`
- trains `stage1_public_small`
- trains `stage2_hardcases_small`
- zips the final checkpoint for download

In [ ]:
# Optional: mount Google Drive to persist outputs
USE_DRIVE = True
DRIVE_DIR = '/content/drive/MyDrive/fact_checking_system_colab'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# Set your repo URL here
REPO_URL = 'https://github.com/injetiharsha/fact_checking_system.git'
BRANCH = 'feat/reduce-heuristics-phased'
REPO_DIR = '/content/fact_checking_system'


In [ ]:
!rm -rf {REPO_DIR}
!git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
# Training-focused dependencies only
!pip install -q --upgrade pip
!pip install -q torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.38.2 datasets==2.17.1 accelerate==0.27.2 scikit-learn==1.4.2 numpy==1.26.4 scipy==1.12.0 PyYAML==6.0.2 sentencepiece==0.2.0 tqdm==4.66.2 requests==2.31.0 urllib3==1.26.18


In [ ]:
import os, torch
print('CUDA available:', torch.cuda.is_available())
print('CUDA device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
os.environ['PYTHONWARNINGS'] = 'ignore'


In [ ]:
# Build the reduced public stage-1 dataset
!python training/common/build_stance_stage1_public.py

In [ ]:
# Stage 1 small
!python -u training/stance/train.py --config training/configs/stance_stage1_public_small.yaml

In [ ]:
# Stage 2 hardcases small
!python -u training/stance/train.py --config training/configs/stance_stage2_hardcases_small.yaml

In [ ]:
# Package final checkpoint and metrics
FINAL_DIR = 'checkpoints/stance/stage2_hardcases_small'
METRICS_DIR = 'training_artifacts/stance/stage2_hardcases_small'
ARCHIVE = '/content/stance_stage2_hardcases_small.zip'
!zip -r {ARCHIVE} {FINAL_DIR} {METRICS_DIR}
print('Created:', ARCHIVE)

In [ ]:
# Optional: copy archive to Drive
if USE_DRIVE:
    import os, shutil
    os.makedirs(DRIVE_DIR, exist_ok=True)
    shutil.copy('/content/stance_stage2_hardcases_small.zip', os.path.join(DRIVE_DIR, 'stance_stage2_hardcases_small.zip'))
    print('Copied archive to', DRIVE_DIR)

In [ ]:
# Download to your machine
from google.colab import files
files.download('/content/stance_stage2_hardcases_small.zip')